# Perceiver CPI — Phase 5 experiments

Trains 4 small models to answer one question: **does it help to choose the model using a validation set made of drugs and proteins the model has never seen?**

| Experiment | Validation set is... |
|---|---|
| `davis_A`, `metz_A` | unseen compounds **and** unseen proteins (built like the test set) |
| `davis_B`, `metz_B` | random pairs, all compounds and proteins already seen (the old way) |

**Before you run this**, attach the data:
1. **Add Input** (top right) → **Datasets** → search for the dataset you uploaded (`perceiver-cpi-data`) → **Add**.
2. **Settings** → **Accelerator** → **GPU T4 x2** (or P100 if your account offers it).
3. **Run All**.

Takes roughly **3–5 hours** on a P100. Kaggle sessions last 12 hours, so this fits in one go. If it does stop early, see the last cell — progress is saved and it continues.

Kaggle gives about 30 GPU hours a week, free.

In [ ]:
# 1. Find the data and copy the code somewhere writable
import os, shutil, subprocess, sys
from pathlib import Path

# On Kaggle these are the real folders; the env vars are only set when testing this notebook offline.
INPUT = Path(os.environ.get('KAGGLE_INPUT', '/kaggle/input'))
WORK = Path(os.environ.get('KAGGLE_WORKING', '/kaggle/working'))

# The dataset may be attached under any name; find it by looking for run_experiment.py.
candidates = sorted(INPUT.glob('*/run_experiment.py')) + sorted(INPUT.glob('*/*/run_experiment.py'))
if not candidates:
    raise FileNotFoundError(
        'Could not find the project data.\n'
        'Use "Add Input" (top right) -> Datasets -> add the dataset you uploaded, then Run All again.\n'
        f'Currently attached: {[p.name for p in INPUT.glob("*")] or "nothing"}')
SOURCE = candidates[0].parent
print('Found the data at:', SOURCE)

# /kaggle/input is read-only, so copy the code across. The split CSVs are read in place.
CODE = WORK / 'perceiver'
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
for item in SOURCE.iterdir():
    (shutil.copytree if item.is_dir() else shutil.copy2)(item, CODE / item.name)
print((CODE / 'BUNDLE_INFO.txt').read_text())

# Results go here. Kaggle saves /kaggle/working as the notebook's output when the run finishes.
RUNS = WORK / 'runs'
RUNS.mkdir(exist_ok=True)
os.environ['PERCEIVER_RUNS'] = str(RUNS)

# If a previous session's output was attached as an input, carry its progress over so training continues.
for previous in sorted(INPUT.glob('*/runs')):
    for folder in previous.iterdir():
        if folder.is_dir() and not (RUNS / folder.name).exists():
            shutil.copytree(folder, RUNS / folder.name)
            print('Continuing a previous run:', folder.name)

In [ ]:
# 2. Show the GPU and install the packages the model needs (2-3 minutes)
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'],
                         capture_output=True, text=True).stdout)
else:
    print('No GPU found. Set Settings -> Accelerator -> GPU T4 x2, then Run All again.')
print('CPU cores:', os.cpu_count())

result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                         'rdkit', 'tape-proteins', 'typed-argument-parser', 'tensorboardX', 'lifelines'],
                        capture_output=True, text=True)
if result.returncode != 0:
    print(result.stdout[-3000:], result.stderr[-3000:])
    raise RuntimeError('Installing packages failed (see above).')

print(subprocess.run([sys.executable, '-c',
                      'import torch; print("PyTorch", torch.__version__, "| GPU:",'
                      ' torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set Accelerator to GPU")'],
                     capture_output=True, text=True).stdout)

In [ ]:
# 3. Quick check that everything works: 1 epoch on the smallest experiment (about 2 minutes)
if any(RUNS.iterdir()):
    print('Resuming a previous session, so the check is skipped.')
    check = None
else:
    check = subprocess.run([sys.executable, 'run_experiment.py', 'davis_A', '--epochs', '1'],
                       cwd=CODE, capture_output=True, text=True,
                       env={**os.environ, 'PERCEIVER_RUNS': str(WORK / 'check')})
print(check.stdout[-2500:], check.stderr[-2500:])
if check.returncode != 0:
    raise RuntimeError('The check failed (see above).')
shutil.rmtree(WORK / 'check', ignore_errors=True)
print('\nCheck passed.')

In [ ]:
# 4. Run the experiments (3-5 hours). Progress prints as it goes.
os.chdir(CODE)
get_ipython().system(f'"{sys.executable}" run_experiment.py {os.environ.get("PERCEIVER_TEST_ARGS", "")}')

In [ ]:
# 5. Compare the results
print(subprocess.run([sys.executable, 'evaluate.py'], cwd=CODE, capture_output=True, text=True).stdout)
results = CODE / 'results.md'
if results.exists():
    shutil.copy2(results, WORK / 'results.md')
    print(results.read_text())

## When it finishes

Everything in `/kaggle/working` is saved as this notebook's output: the trained models, the predictions and `results.md`.

**Download `results.md` and the `runs` folder** (Output tab, right-hand side), or just send `results.md` to Claude.

**If the session stopped before it finished:** save the version, then start a new run and add *this notebook's output* as an input dataset. Cell 1 picks up the saved progress and carries on from the last finished epoch.